# Basic Usage - dask_setup Recipe

This notebook demonstrates the fundamental patterns for using dask_setup, including different workload types, resource management, and cluster configuration.

## Key Learning Points
- CPU, I/O, and mixed workload configurations
- Resource detection and memory management
- Dashboard access and monitoring
- Safe cluster shutdown patterns
- HPC environment integration

## Setup and Imports

First, let's import the necessary libraries and set up our environment.

In [1]:
# Core imports
import sys
import time
from pathlib import Path
import warnings

# Add dask_setup to path (adjust if needed for your environment)
import sys
sys.path.insert(0, str(Path.cwd().parent.parent.parent / "src"))
sys.path.insert(0, str(Path.cwd().parent))

# Scientific computing imports
import numpy as np
import dask
import dask.array as da
import xarray as xr

# dask_setup import
try:
    from dask_setup import setup_dask_client
    print("dask_setup imported successfully")
except ImportError as e:
    print(f"Failed to import dask_setup: {e}")
    print("Please ensure dask_setup is installed: pip install -e .")
    
# Utility imports (from recipes directory)
try:
    from utils import format_bytes, format_duration, timer
    print("utils imported successfully")
except ImportError as e:
    print(f"utils not available: {e}")
    # Define minimal versions if utils not available
    def format_bytes(bytes_val):
        return f"{bytes_val / (1024**3):.2f} GB"
    def format_duration(seconds):
        return f"{seconds:.2f}s"

# Optional imports for better display
try:
    import psutil
    PSUTIL_AVAILABLE = True
    print("psutil available for system monitoring")
except ImportError:
    PSUTIL_AVAILABLE = False
    print("psutil not available - system monitoring limited")

print("\nEnvironment setup complete!")

dask_setup imported successfully
utils not available: cannot import name 'format_bytes' from 'utils' (/Users/green/GitHub/dask_setup/examples/recipes/utils.py)
psutil available for system monitoring

Environment setup complete!


## System Information

Let's check our system resources and environment before setting up dask clusters.

In [4]:
import os

print("SYSTEM INFORMATION")
print("=" * 50)

# Basic system info
if PSUTIL_AVAILABLE:
    cpu_count = psutil.cpu_count(logical=False)  # Physical cores
    cpu_count_logical = psutil.cpu_count(logical=True)  # Including hyperthreading
    memory = psutil.virtual_memory()
    
    print(f"CPU cores (physical): {cpu_count}")
    print(f"CPU cores (logical):  {cpu_count_logical}")
    print(f"Total memory: {format_bytes(memory.total)}")
    print(f"Available memory: {format_bytes(memory.available)}")
    print(f"Memory usage: {memory.percent:.1f}%")
else:
    print("System info limited (psutil not available)")

# Check for HPC environment variables
print("\n  HPC ENVIRONMENT DETECTION")
print("=" * 50)

hpc_vars = {
    'PBS_JOBID': 'PBS Job ID',
    'SLURM_JOB_ID': 'SLURM Job ID',
    'PBS_NCPUS': 'PBS CPU count',
    'SLURM_CPUS_ON_NODE': 'SLURM CPU count',
    'PBS_MEM': 'PBS Memory allocation',
    'SLURM_MEM_PER_NODE': 'SLURM Memory per node',
    'PBS_JOBFS': 'PBS fast local storage',
    'TMPDIR': 'Temporary directory'
}

hpc_detected = False
for var, description in hpc_vars.items():
    value = os.environ.get(var)
    if value:
        print(f" {description}: {value}")
        hpc_detected = True

if not hpc_detected:
    print("ℹ  No HPC environment detected (running on local system)")
    print("   This is normal for local development")

print("\n System check complete")

SYSTEM INFORMATION
CPU cores (physical): 10
CPU cores (logical):  10
Total memory: 16.00 GB
Available memory: 3.85 GB
Memory usage: 75.9%

  HPC ENVIRONMENT DETECTION
 Temporary directory: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/

 System check complete


## CPU-Intensive Workload

CPU-intensive workloads benefit from process-based parallelism where each worker runs in a separate process with one thread. This avoids Python's GIL (Global Interpreter Lock) limitations.

In [9]:
print(" CPU-INTENSIVE WORKLOAD DEMONSTRATION")
print("=" * 60)

# Create CPU-optimized cluster
client, cluster, temp_dir = setup_dask_client(
    workload_type="cpu",
    max_workers=4,  # Adjust based on your system
    reserve_mem_gb=2.0,
    dashboard=True
)

print(f" CPU cluster created")
print(f"   Workers: {len(client.scheduler_info()['workers'])}")
print(f"   Temp directory: {temp_dir}")
print(f"   Dashboard: {client.dashboard_link}")

# Display cluster information
worker_info = client.scheduler_info()['workers']
if worker_info:
    first_worker = next(iter(worker_info.values()))
    print(f"   Memory per worker: ~{first_worker.get('memory_limit', 0) / (1024**3):.1f} GB")
    print(f"   Threads per worker: {first_worker.get('nthreads', 1)}")

# Create sample CPU-intensive computation
print("\n Running CPU-intensive computation...")

# Create a moderately large array for computation
x = da.random.random((5000, 5000), chunks=(1000, 1000))
print(f"   Array shape: {x.shape}")
print(f"   Array chunks: {x.chunks}")
#print(f"   Number of chunks: {x.nchunks}")

# Time the computation
start_time = time.time()
result = (x + x.T).mean().compute()  # CPU-intensive: transpose and add
computation_time = time.time() - start_time

print(f"   Computation result: {result:.6f}")
print(f"   Computation time: {format_duration(computation_time)}")
print(f"   Throughput: {format_bytes(x.nbytes)} in {format_duration(computation_time)} = {format_bytes(x.nbytes/computation_time)}/s")

 CPU-INTENSIVE WORKLOAD DEMONSTRATION
Dask dashboard: http://127.0.0.1:54110/status
Tunnel from your laptop (run locally):
  ssh -N -L 8787:Sams-MacBook-Pro.local:54110 gadi.nci.org.au
Then open: http://localhost:8787

[setup_dask_client] temp/spill dir: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/dask-72382/dask-72382/dask-72382/dask-72382\nWorkers: 4 | threads/worker: 1 | processes: True\nMem: total ~16.0 GiB | usable ~14.0 GiB | per-worker ~3.5 GiB\nCompression: spill=auto | comm=False
 CPU cluster created
   Workers: 4
   Temp directory: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/dask-72382/dask-72382/dask-72382/dask-72382
   Dashboard: http://127.0.0.1:54110/status
   Memory per worker: ~3.5 GB
   Threads per worker: 1

 Running CPU-intensive computation...
   Array shape: (5000, 5000)
   Array chunks: ((1000, 1000, 1000, 1000, 1000), (1000, 1000, 1000, 1000, 1000))
   Computation result: 1.000005
   Computation time: 0.88s
   Throughput: 0.19 GB in 0.88s = 0.21 GB/s


In [10]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:54110/status,
Dashboard: http://127.0.0.1:54110/status,Workers: 4
Total threads: 4,Total memory: 14.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54111,Workers: 0
Dashboard: http://127.0.0.1:54110/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:54122,Total threads: 1
Dashboard: http://127.0.0.1:54124/status,Memory: 3.50 GiB
Nanny: tcp://127.0.0.1:54114,


In [11]:
# Clean up
client.close()
cluster.close()
print("\n CPU cluster cleaned up")


 CPU cluster cleaned up


## I/O-Intensive Workload

I/O-intensive workloads benefit from thread-based parallelism with fewer processes but more threads per process. This is ideal for file operations, network requests, and database queries.

In [13]:
print(" I/O-INTENSIVE WORKLOAD DEMONSTRATION")
print("=" * 60)

# Create I/O-optimized cluster  
client, cluster, temp_dir = setup_dask_client(
    workload_type="io",
    max_workers=2,  # Fewer workers, more threads each
    reserve_mem_gb=2.0,
    dashboard=True
)

print(f" I/O cluster created")
print(f"   Workers: {len(client.scheduler_info()['workers'])}")
print(f"   Temp directory: {temp_dir}")
print(f"   Dashboard: {client.dashboard_link}")

# Display cluster information
worker_info = client.scheduler_info()['workers']
if worker_info:
    first_worker = next(iter(worker_info.values()))
    print(f"   Memory per worker: ~{first_worker.get('memory_limit', 0) / (1024**3):.1f} GB")
    print(f"   Threads per worker: {first_worker.get('nthreads', 1)}")

# Simulate I/O-intensive work by creating and processing multiple smaller arrays
print("\n Running I/O-intensive simulation...")

# Create multiple smaller arrays (simulating reading many files)
arrays = []
for i in range(20):  # Simulate 20 files
    arr = da.random.random((500, 500), chunks=(250, 250))
    arrays.append(arr)

print(f"   Created {len(arrays)} arrays (simulating file reads)")
print(f"   Each array shape: {arrays[0].shape}")

# Process all arrays (simulating file processing)
start_time = time.time()

# Compute means of all arrays (I/O-like pattern)
means = [arr.mean() for arr in arrays]
results = da.stack(means).compute()

computation_time = time.time() - start_time

print(f"   Processed {len(arrays)} arrays")
print(f"   Mean of means: {results.mean():.6f}")
print(f"   Processing time: {format_duration(computation_time)}")
print(f"   Throughput: {len(arrays)} arrays in {format_duration(computation_time)} = {len(arrays)/computation_time:.1f} arrays/s")

 I/O-INTENSIVE WORKLOAD DEMONSTRATION
Dask dashboard: http://192.168.4.113:54189/status
Tunnel from your laptop (run locally):
  ssh -N -L 8787:Sams-MacBook-Pro.local:54189 gadi.nci.org.au
Then open: http://localhost:8787

[setup_dask_client] temp/spill dir: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382\nWorkers: 1 | threads/worker: 5 | processes: False\nMem: total ~16.0 GiB | usable ~14.0 GiB | per-worker ~14.0 GiB\nCompression: spill=auto | comm=False
 I/O cluster created
   Workers: 1
   Temp directory: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382
   Dashboard: http://192.168.4.113:54189/status
   Memory per worker: ~14.0 GB
   Threads per worker: 5

 Running I/O-intensive simulation...
   Created 20 arrays (simulating file reads)
   Each array shape: (500, 500)
   Processed 20 arrays
   Mean of means: 0.500179
   Processing time: 0.12s
   Throug

In [14]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://192.168.4.113:54189/status,
Dashboard: http://192.168.4.113:54189/status,Workers: 1
Total threads: 5,Total memory: 14.00 GiB
Status: running,Using processes: False
Comm: inproc://192.168.4.113/72382/1,Workers: 0
Dashboard: http://192.168.4.113:54189/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: inproc://192.168.4.113/72382/4,Total threads: 5
Dashboard: http://192.168.4.113:54190/status,Memory: 14.00 GiB
Nanny: None,


In [15]:
# Clean up
client.close()
cluster.close()
print("\n I/O cluster cleaned up")


 I/O cluster cleaned up


## Mixed Workload

Mixed workloads balance between CPU and I/O operations. This configuration provides a compromise between the two extremes.

In [ ]:
print("  MIXED WORKLOAD DEMONSTRATION")
print("=" * 60)

# Create mixed workload cluster
client, cluster, temp_dir = setup_dask_client(
    workload_type="mixed",
    max_workers=3,
    reserve_mem_gb=20.0,
    dashboard=True
)

print(f"✅ Mixed cluster created")
print(f"   Workers: {len(client.scheduler_info()['workers'])}")
print(f"   Temp directory: {temp_dir}")
print(f"   Dashboard: {client.dashboard_link}")

# Display cluster information
worker_info = client.scheduler_info()['workers']
if worker_info:
    first_worker = next(iter(worker_info.values()))
    print(f"   Memory per worker: ~{first_worker.get('memory_limit', 0) / (1024**3):.1f} GB")
    print(f"   Threads per worker: {first_worker.get('nthreads', 1)}")

# Demonstrate mixed computation and "I/O"
print("\n Running mixed workload...")

# Step 1: "Load" data (simulate reading multiple datasets)
datasets = []
for i in range(5):
    data = da.random.random((1000, 1000), chunks=(500, 500))
    datasets.append(data)

print(f"   'Loaded' {len(datasets)} datasets")

# Step 2: Process each dataset (CPU-intensive)
start_time = time.time()

processed_results = []
for i, dataset in enumerate(datasets):
    # Simulate some computation on each dataset
    result = (dataset * 2 + 1).mean()  # Simple computation
    processed_results.append(result)
    print(f"   Processed dataset {i+1}/{len(datasets)}")

# Step 3: Combine results
final_result = da.stack(processed_results).mean().compute()

computation_time = time.time() - start_time

print(f"   Final result: {final_result:.6f}")
print(f"   Total processing time: {format_duration(computation_time)}")

# Clean up
client.close()
cluster.close()
print("\n🧹 Mixed cluster cleaned up")

## Monitoring and Dashboard Usage

The Dask dashboard is crucial for monitoring cluster performance and debugging issues. Let's explore how to use it effectively.

In [20]:
print(" DASHBOARD AND MONITORING DEMONSTRATION")
print("=" * 60)

# Create a cluster for monitoring demonstration
client, cluster, temp_dir = setup_dask_client(
    workload_type="cpu",
    max_workers=2,
    reserve_mem_gb=2.0,
    dashboard=True
)

print(f" Monitoring cluster created")
print(f" Dashboard URL: {client.dashboard_link}")
print(f"    Click the link above to open the dashboard in a new tab")

# Show scheduler info
scheduler_info = client.scheduler_info()
print(f"\n  Cluster Status:")
print(f"   Scheduler address: {scheduler_info['address']}")
print(f"   Workers: {len(scheduler_info['workers'])}")

for worker_id, worker_info in scheduler_info['workers'].items():
    print(f"   Worker {worker_id[:8]}:")
    #print(f"     Address: {worker_info['address']}")
    print(f"     Threads: {worker_info['nthreads']}")
    print(f"     Memory: {format_bytes(worker_info['memory_limit'])}")

# Run a computation while dashboard is available
print(f"\n Running computation (monitor in dashboard)...")
print(f"    In the dashboard, go to:")
print(f"      - Status: Overall cluster health")
print(f"      - Task Stream: Real-time task execution")
print(f"      - Memory: Worker memory usage")
print(f"      - Workers: Individual worker status")

# Create a longer-running computation for dashboard observation
large_array = da.random.random((8000, 8000), chunks=(1000, 1000))
print(f"   Array size: {format_bytes(large_array.nbytes)}")
print(f"   Chunks: {large_array.chunks}")

# Run computation
start_time = time.time()
result = large_array.sum().compute()
computation_time = time.time() - start_time

print(f"   Result: {result:.2e}")
print(f"   Time: {format_duration(computation_time)}")
print(f"\n    Check the dashboard to see the task execution pattern!")

# Keep cluster alive for a moment to allow dashboard inspection
print(f"\n⏱  Cluster will remain active for 10 seconds for dashboard inspection...")
time.sleep(10)

# Clean up
client.close()
cluster.close()
print("\n Monitoring cluster cleaned up")

 DASHBOARD AND MONITORING DEMONSTRATION
Dask dashboard: http://127.0.0.1:54358/status
Tunnel from your laptop (run locally):
  ssh -N -L 8787:Sams-MacBook-Pro.local:54358 gadi.nci.org.au
Then open: http://localhost:8787

[setup_dask_client] temp/spill dir: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382\nWorkers: 2 | threads/worker: 1 | processes: True\nMem: total ~16.0 GiB | usable ~14.0 GiB | per-worker ~7.0 GiB\nCompression: spill=auto | comm=False
 Monitoring cluster created
 Dashboard URL: http://127.0.0.1:54358/status
    Click the link above to open the dashboard in a new tab

  Cluster Status:
   Scheduler address: tcp://127.0.0.1:54359
   Workers: 2
   Worker tcp://12:
     Threads: 1
     Memory: 7.00 GB
   Worker tcp://12:
     Threads: 1
     Memory: 7.00 GB

 Running computation (monitor in dashboard)...
    In the dashboard, go to:
      - Status: Over

2026-02-25 10:03:06,394 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/Users/green/miniforge3/envs/napari-env/lib/python3.13/site-packages/distributed/comm/tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/green/miniforge3/envs/napari-env/lib/python3.13/site-packages/distributed/worker.py", line 1267, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
    ...<14 lines>...
    )
    ^
  File "/Users/green/miniforge3/envs/napari-env/lib/python3.13/site-packages/distributed/utils_comm.py", line 416, in retry_operation
    return await retry(
           ^^^^^^^^^^^^
    ...<5 lines>...
    )
 

## Xarray Integration

dask_setup works excellently with xarray for scientific data analysis. Let's see how to use them together.

In [21]:
print(" XARRAY INTEGRATION DEMONSTRATION")
print("=" * 60)

# Create cluster for xarray work
client, cluster, temp_dir = setup_dask_client(
    workload_type="mixed",  # Good for xarray's mixed I/O and computation
    max_workers=3,
    reserve_mem_gb=2.0,
    dashboard=True
)

print(f" Xarray cluster created")
print(f"   Dashboard: {client.dashboard_link}")

# Create synthetic climate-like dataset
print(f"\n Creating synthetic climate dataset...")

# Create coordinates
time = np.arange('2000-01-01', '2005-01-01', dtype='datetime64[D]')
lat = np.linspace(-90, 90, 180)
lon = np.linspace(-180, 180, 360)

print(f"   Time steps: {len(time)}")
print(f"   Spatial grid: {len(lat)} × {len(lon)}")

# Create dask arrays for the data
# Temperature with seasonal cycle
temp_data = da.random.random((len(time), len(lat), len(lon)), chunks=(365, 90, 180))
temp_data = temp_data * 50 - 10 + 273.15  # Realistic temperature range in Kelvin

# Add seasonal cycle
day_of_year = np.arange(len(time)) % 365
seasonal_cycle = 10 * np.cos(2 * np.pi * day_of_year / 365.25)
temp_data = temp_data + seasonal_cycle[:, None, None]

# Create xarray dataset
ds = xr.Dataset({
    'temperature': (['time', 'lat', 'lon'], temp_data, {
        'units': 'K',
        'long_name': 'Near-surface air temperature'
    })
}, coords={
    'time': time,
    'lat': ('lat', lat, {'units': 'degrees_north'}),
    'lon': ('lon', lon, {'units': 'degrees_east'})
})

print(f"   Dataset created:")
print(f"     Shape: {ds.temperature.shape}")
print(f"     Chunks: {ds.temperature.chunks}")
print(f"     Size: {format_bytes(ds.temperature.nbytes)}")

# Demonstrate typical xarray operations
print(f"\n Running xarray computations...")

# 1. Global mean temperature time series
start_time = time.time()
global_mean = ds.temperature.mean(['lat', 'lon'])
global_mean_computed = global_mean.compute()
time1 = time.time() - start_time

print(f"   1. Global mean time series: {format_duration(time1)}")
print(f"      Result shape: {global_mean_computed.shape}")
print(f"      Mean temperature: {global_mean_computed.mean().values:.2f} K")

# 2. Seasonal climatology
start_time = time.time()
seasonal_clim = ds.temperature.groupby('time.season').mean('time')
seasonal_computed = seasonal_clim.compute()
time2 = time.time() - start_time

print(f"   2. Seasonal climatology: {format_duration(time2)}")
print(f"      Seasons: {list(seasonal_computed.season.values)}")

# 3. Annual mean
start_time = time.time()
annual_mean = ds.temperature.resample(time='1Y').mean()
annual_computed = annual_mean.compute()
time3 = time.time() - start_time

print(f"   3. Annual means: {format_duration(time3)}")
print(f"      Years: {len(annual_computed.time)}")

total_time = time1 + time2 + time3
print(f"\n   Total computation time: {format_duration(total_time)}")
print(f"   Data processed: {format_bytes(ds.temperature.nbytes)}")
print(f"   Throughput: {format_bytes(ds.temperature.nbytes / total_time)}/s")

# Clean up
client.close()
cluster.close()
print("\n🧹 Xarray cluster cleaned up")

 XARRAY INTEGRATION DEMONSTRATION
Dask dashboard: http://127.0.0.1:54460/status
Tunnel from your laptop (run locally):
  ssh -N -L 8787:Sams-MacBook-Pro.local:54460 gadi.nci.org.au
Then open: http://localhost:8787

[setup_dask_client] temp/spill dir: /var/folders/30/ng9_tlys2f1802_03ng09n0m0000gn/T/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382/dask-72382\nWorkers: 3 | threads/worker: 2 | processes: True\nMem: total ~16.0 GiB | usable ~14.0 GiB | per-worker ~4.7 GiB\nCompression: spill=auto | comm=False
 Xarray cluster created
   Dashboard: http://127.0.0.1:54460/status

 Creating synthetic climate dataset...
   Time steps: 1827
   Spatial grid: 180 × 360
   Dataset created:
     Shape: (1827, 180, 360)
     Chunks: ((365, 365, 365, 365, 365, 2), (90, 90), (180, 180))
     Size: 0.88 GB

 Running xarray computations...


AttributeError: 'numpy.ndarray' object has no attribute 'time'

## Best Practices and Tips

Here are some key best practices when using dask_setup:

In [ ]:
print("💡 BEST PRACTICES AND TIPS")
print("=" * 60)

print("\n1. 🎯 WORKLOAD TYPE SELECTION:")
print("   • CPU-intensive: Use 'cpu' for heavy computation (NumPy, SciPy)")
print("   • I/O-intensive: Use 'io' for file operations, web requests")
print("   • Mixed: Use 'mixed' for balanced workflows")

print("\n2. 💾 MEMORY MANAGEMENT:")
print("   • Reserve adequate memory (30-40% of total)")
print("   • Monitor memory usage in dashboard")
print("   • Use appropriate chunk sizes (256-512 MB)")

print("\n3. 🔧 CHUNK SIZE OPTIMIZATION:")
print("   • Too small: High overhead, poor performance")
print("   • Too large: Memory issues, poor parallelism")
print("   • Target: 256-512 MB per chunk")

# Demonstrate chunk size impact
print("\n🧪 Chunk size demonstration:")

# Create small cluster for demo
client, cluster, temp_dir = setup_dask_client(
    workload_type="cpu",
    max_workers=2,
    reserve_mem_gb=5.0,
    dashboard=False
)

# Test different chunk sizes
array_shape = (4000, 4000)
chunk_sizes = [(100, 100), (1000, 1000), (2000, 2000)]

for chunks in chunk_sizes:
    # Create array
    arr = da.random.random(array_shape, chunks=chunks)
    
    # Estimate chunk size
    chunk_bytes = chunks[0] * chunks[1] * 8  # 8 bytes per float64
    chunk_mb = chunk_bytes / (1024**2)
    
    # Time computation
    start_time = time.time()
    result = arr.mean().compute()
    comp_time = time.time() - start_time
    
    print(f"   Chunks {chunks}: {chunk_mb:.1f} MB each, {arr.nchunks} chunks, {format_duration(comp_time)}")

print("\n4. 📊 DASHBOARD USAGE:")
print("   • Status: Overall cluster health")
print("   • Task Stream: Task execution timeline")
print("   • Memory: Worker memory usage")
print("   • Progress: Long-running operation status")

print("\n5. 🖥️  HPC CONSIDERATIONS:")
print("   • Use SSH tunnels for dashboard access")
print("   • Leverage $PBS_JOBFS for temporary storage")
print("   • Monitor job resource usage")
print("   • Plan for job time limits")

print("\n6. 🚨 COMMON PITFALLS:")
print("   • Don't forget to close clusters (client.close(), cluster.close())")
print("   • Watch out for memory leaks in long-running computations")
print("   • Be careful with very large arrays that exceed memory")
print("   • Test with small data before scaling up")

client.close()
cluster.close()

print("\n✅ Best practices demonstration complete")

## Troubleshooting Common Issues

Here are solutions to common problems you might encounter:

In [22]:
print("🔧 TROUBLESHOOTING COMMON ISSUES")
print("=" * 60)

print("\n❌ Problem: 'Task requires > memory_limit'")
print("✅ Solutions:")
print("   • Reduce chunk sizes")
print("   • Increase reserve_mem_gb")
print("   • Use fewer workers with more memory each")
print("   Example: setup_dask_client('cpu', max_workers=1, reserve_mem_gb=60)")

print("\n❌ Problem: Dashboard not accessible on HPC")
print("✅ Solutions:")
print("   • Use SSH tunnel: ssh -N -L 8787:compute-node:port login-node")
print("   • Check firewall settings")
print("   • Ensure dashboard=True in setup_dask_client()")

print("\n❌ Problem: Slow performance")
print("✅ Solutions:")
print("   • Check chunk sizes (aim for 256-512 MB)")
print("   • Verify workload_type matches your use case")
print("   • Monitor dashboard for bottlenecks")
print("   • Consider data locality and access patterns")

print("\n❌ Problem: Out of memory errors")
print("✅ Solutions:")
print("   • Increase spill thresholds")
print("   • Use smaller chunk sizes")
print("   • Process data in batches")
print("   • Monitor memory usage in dashboard")

print("\n❌ Problem: Workers frequently restarting")
print("✅ Solutions:")
print("   • Check system resources (memory, disk space)")
print("   • Reduce memory pressure")
print("   • Check for resource contention")
print("   • Monitor system logs")

print("\n💡 Debugging Tips:")
print("   • Use verbose logging for detailed information")
print("   • Start with small test cases")
print("   • Monitor the dashboard during execution")
print("   • Check cluster.logs() for worker messages")
print("   • Use client.profile() for performance analysis")

🔧 TROUBLESHOOTING COMMON ISSUES

❌ Problem: 'Task requires > memory_limit'
✅ Solutions:
   • Reduce chunk sizes
   • Increase reserve_mem_gb
   • Use fewer workers with more memory each
   Example: setup_dask_client('cpu', max_workers=1, reserve_mem_gb=60)

❌ Problem: Dashboard not accessible on HPC
✅ Solutions:
   • Use SSH tunnel: ssh -N -L 8787:compute-node:port login-node
   • Check firewall settings
   • Ensure dashboard=True in setup_dask_client()

❌ Problem: Slow performance
✅ Solutions:
   • Check chunk sizes (aim for 256-512 MB)
   • Verify workload_type matches your use case
   • Monitor dashboard for bottlenecks
   • Consider data locality and access patterns

❌ Problem: Out of memory errors
✅ Solutions:
   • Increase spill thresholds
   • Use smaller chunk sizes
   • Process data in batches
   • Monitor memory usage in dashboard

❌ Problem: Workers frequently restarting
✅ Solutions:
   • Check system resources (memory, disk space)
   • Reduce memory pressure
   • Check for 

## Conclusion

You've now learned the fundamentals of using dask_setup! Here's a summary of what we covered:

### Key Takeaways:
1. **Workload Types**: Choose `cpu`, `io`, or `mixed` based on your computation pattern
2. **Resource Management**: Reserve adequate memory and monitor usage
3. **Dashboard**: Essential for monitoring and debugging
4. **Chunk Sizes**: Target 256-512 MB for optimal performance
5. **Integration**: Works seamlessly with xarray and other scientific libraries

### Next Steps:
- Explore other recipes for more advanced topics
- Try dask_setup with your own datasets
- Experiment with different configurations
- Check out the dashboard features in detail

### Additional Resources:
- dask_setup documentation and examples
- Dask documentation: https://docs.dask.org/
- Xarray documentation: https://xarray.pydata.org/

Happy computing with dask_setup! 🚀

In [2]:
from dask_setup import recommend_io_chunks

In [3]:
path = "~/GitHub/zarr-dump/oisst_aus_2015-2025.zarr/"

In [4]:
ds = xr.open_zarr(path, consolidated=False, zarr_format=3)

/Users/green/miniforge3/envs/napari-env/lib/python3.13/site-packages/zarr/core/group.py:3530: ZarrUserWarning: Object at .DS_Store is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [6]:
ds

<xarray.Dataset> Size: 5GB
Dimensions:  (time: 3889, zlev: 1, lat: 240, lon: 360)
Coordinates:
  * time     (time) datetime64[ns] 31kB 2015-01-01T12:00:00 ... 2025-08-24T12...
  * lon      (lon) float32 1kB 90.12 90.38 90.62 90.88 ... 179.4 179.6 179.9
  * lat      (lat) float32 960B -59.88 -59.62 -59.38 ... -0.625 -0.375 -0.125
Dimensions without coordinates: zlev
Data variables:
    err      (time, zlev, lat, lon) float32 1GB dask.array<chunksize=(64, 1, 240, 360), meta=np.ndarray>
    ice      (time, zlev, lat, lon) float32 1GB dask.array<chunksize=(64, 1, 240, 360), meta=np.ndarray>
    anom     (time, zlev, lat, lon) float32 1GB dask.array<chunksize=(64, 1, 240, 360), meta=np.ndarray>
    sst      (time, zlev, lat, lon) float32 1GB dask.array<chunksize=(64, 1, 240, 360), meta=np.ndarray>
Attributes: (12/40)
    CDI:                        Climate Data Interface version 2.4.1 (https:/...
    Conventions:                CF-1.6, ACDD-1.3
    source:                     ICOADS, NCEP_GTS, GSFC_ICE, NCEP_ICE, Pathfin...
    institution:                NOAA/National Centers for Environmental Infor...
    title:                      NOAA/NCEI 1/4 Degree Daily Optimum Interpolat...
    id:                         oisst-avhrr-v02r01.19810901.nc
    ...                         ...
    comment:                    Data was converted from NetCDF-3 to NetCDF-4 ...
    sensor:                     Thermometer, AVHRR
    references:                 Reynolds, et al.(2007) Daily High-Resolution-...
    CDO:                        Climate Data Operators version 2.4.1 (https:/...
    NCO:                        netCDF Operators version 5.3.3 (Homepage = ht...
    history:                    Tue Aug 26 13:05:33 2025: ncatted -O -a histo...

In [5]:
# Automatic format detection and optimization
chunks = recommend_io_chunks(
    ds,
    path_or_url=path, 
    storage_location="local",           # Optimized for local NVMe/SSD
    access_pattern="random",
    verbose=True                          # Random access pattern
)

 I/O Optimization Recommendations
 Format: ZARR
 Location: local
 Access pattern: random
 Recommended chunks: {'lon': 90}
 Estimated chunk size: 320.4 MiB
 Compression: blosc (level 5)
⚡ Estimated throughput: 150.0 MB/s

 Usage:
   ds_chunked = ds.chunk({'lon': 90})
